<a href="https://colab.research.google.com/github/elshaymaAnalyst/PythonProject/blob/main/Pyspark_Basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Fetched 6,555 B in 2s (3,830 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
62 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


In [3]:
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz



In [4]:
!tar xf spark-3.2.1-bin-hadoop3.2.tgz

In [5]:
!pip install -q findspark


In [6]:
!pip install pyspark


In [7]:
!pip install py4j

In [8]:
import os
import sys

In [9]:
import findspark
findspark.init()
findspark.find()

'/usr/local/lib/python3.12/dist-packages/pyspark'

In [10]:
from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

In [11]:
df = spark.read.csv("/content/Pima  (1).csv", header=True, inferSchema=True)


In [12]:
df.columns

['preg_count',
 'glucose_concentration',
 'diastolic_bp',
 'triceps_skin_fold_thickness',
 'two_hr_serum_insulin',
 'bmi',
 'diabetes_pedi',
 'age',
 'diabetes_class']

In [13]:
from pyspark.ml.feature import VectorAssembler

In [14]:
Assembler = VectorAssembler(inputCols=['preg_count',
 'glucose_concentration',
 'diastolic_bp',
 'triceps_skin_fold_thickness',
 'two_hr_serum_insulin',
 'bmi',
 'diabetes_pedi',
 'age'],outputCol='features')


In [15]:
df = Assembler.transform(df)

In [16]:
df.show(2)

+----------+---------------------+------------+---------------------------+--------------------+----+-------------+---+--------------+--------------------+
|preg_count|glucose_concentration|diastolic_bp|triceps_skin_fold_thickness|two_hr_serum_insulin| bmi|diabetes_pedi|age|diabetes_class|            features|
+----------+---------------------+------------+---------------------------+--------------------+----+-------------+---+--------------+--------------------+
|         6|                  148|          72|                         35|                   0|33.6|        0.627| 50|             1|[6.0,148.0,72.0,3...|
|         1|                   85|          66|                         29|                   0|26.6|        0.351| 31|             0|[1.0,85.0,66.0,29...|
+----------+---------------------+------------+---------------------------+--------------------+----+-------------+---+--------------+--------------------+
only showing top 2 rows



In [19]:
df=df.select(['features','diabetes_class'])

In [20]:
df.show(2)

+--------------------+--------------+
|            features|diabetes_class|
+--------------------+--------------+
|[6.0,148.0,72.0,3...|             1|
|[1.0,85.0,66.0,29...|             0|
+--------------------+--------------+
only showing top 2 rows



In [21]:
(train_data, test_data) = df.randomSplit([0.80, 0.20])

In [22]:
from pyspark.ml.classification import LogisticRegression

In [23]:
lrm=LogisticRegression(featuresCol='features',labelCol='diabetes_class')

In [24]:
lrm=lrm.fit(train_data)

In [25]:
predictions=lrm.transform(test_data)

In [26]:
predictions.show(2)

+--------------------+--------------+--------------------+--------------------+----------+
|            features|diabetes_class|       rawPrediction|         probability|prediction|
+--------------------+--------------+--------------------+--------------------+----------+
|(8,[0,1,6,7],[6.0...|             0|[3.46784954228946...|[0.96975901685737...|       0.0|
|(8,[0,1,6,7],[10....|             1|[2.91665754645429...|[0.94866376272460...|       0.0|
+--------------------+--------------+--------------------+--------------------+----------+
only showing top 2 rows



In [27]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [28]:
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol='diabetes_class', predictionCol='prediction', metricName='accuracy')
accuracy = evaluator_accuracy.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7091
